In [ ]:
%cd /run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough

In [ ]:
import argparse
import importlib.util
import inspect
import json
import math
import os
import pickle
import random
import shutil
import socket
import subprocess
import sys
import tempfile
import warnings
from pathlib import Path

import joblib
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy import signal, stats
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, auc, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_recall_curve, precision_score,
    recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import (
    GroupShuffleSplit, StratifiedGroupKFold, StratifiedKFold,
    cross_val_score, train_test_split
)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils import resample
from tensorboard.backend.event_processing import event_accumulator
from torch.amp import GradScaler
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
from transformers import (
    AutoConfig, AutoFeatureExtractor, AutoModel,
    get_linear_schedule_with_warmup
)

# Custom modules
import commons
import models
import utils
from augmentation import DataAugmentator, ISD_additive_noise, LnL_convolutive_noise
from cough_datasets import CoughDatasets, CoughDatasetsCollate
from wrapper.wav2vec import Wav2VecWrapper
from wrapper.wavlm_plus import WavLMWrapper
from wrapper.whisper import WhisperWrapper

import xgboost as xgb
import shap

data_augmentator = DataAugmentator(None, "/run/media/fourier/Data1/Pras/Interspeech2025/RIRS_NOISES/data_augmentation_noises_labels.tsv",
                None, "/run/media/fourier/Data1/Pras/Interspeech2025/RIRS_NOISES/data_augmentation_rirs_labels.tsv",
                5.5, ["apply_speed_perturbation", "apply_reverb", "add_background_noise", "apply_pitch_shift", "apply_random_gain"])

import warnings
warnings.simplefilter("ignore", UserWarning)

In [ ]:
def load_config():
    return {
        "sr": 16000,
        "bp_low": 50,
        "bp_high": 8000,
        "frame_ms": 25,
        "hop_ms": 10,
        "mel_bins": 64,
        "mfcc": 13,
        "quantile_mapping": [20, 40, 60, 80],
    }

def extract_envelope_features(audio, sr):
    # Compute envelope
    envelope = np.abs(librosa.util.frame(audio, frame_length=int(0.025*sr), hop_length=int(0.01*sr)))
    envelope = np.mean(envelope, axis=0)
    
    # Find peaks in envelope
    peaks, _ = signal.find_peaks(envelope, height=np.max(envelope) * 0.3)
    
    # Inter-pulse intervals
    if len(peaks) > 1:
        intervals = np.diff(peaks) / sr * 0.01  # Convert to seconds (frame hop)
        inter_pulse_median = np.median(intervals)
    else:
        inter_pulse_median = 0
    
    # Attack and decay times
    max_idx = np.argmax(envelope)
    attack_frames = 0
    decay_frames = 0
    
    # Find attack time (10% to 90% of peak)
    peak_val = envelope[max_idx]
    thresh_10 = peak_val * 0.1
    thresh_90 = peak_val * 0.9
    
    for i in range(max_idx):
        if envelope[i] >= thresh_10:
            attack_start = i
            break
    else:
        attack_start = 0
    
    for i in range(attack_start, max_idx):
        if envelope[i] >= thresh_90:
            attack_frames = i - attack_start
            break
    
    # Find decay time (peak to 50%)
    thresh_50 = peak_val * 0.5
    for i in range(max_idx, len(envelope)):
        if envelope[i] <= thresh_50:
            decay_frames = i - max_idx
            break
    else:
        decay_frames = len(envelope) - max_idx
    
    # Convert frames to milliseconds
    frame_to_ms = 10  # 10ms hop
    attack_ms = attack_frames * frame_to_ms
    decay_ms = decay_frames * frame_to_ms
    
    return {
        'pulse_count': len(peaks),
        'inter_pulse_median': inter_pulse_median,
        'attack_ms': attack_ms,
        'decay_ms': decay_ms,
        'attack_decay_ratio': attack_ms / (decay_ms + 1e-8),
        'envelope_mod_depth': np.std(envelope) / (np.mean(envelope) + 1e-8)
    }

def extract_spectral_features(audio, sr, config):
    # Spectral features
    spectral_centroids = librosa.feature.spectral_centroid(y=audio, sr=sr)[0]
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=audio, sr=sr)[0]
    spectral_rolloff = librosa.feature.spectral_rolloff(y=audio, sr=sr, roll_percent=0.85)[0]
    spectral_flatness = librosa.feature.spectral_flatness(y=audio)[0]
    spectral_contrast = librosa.feature.spectral_contrast(y=audio, sr=sr)[0]
    
    # Zero crossing rate
    zcr = librosa.feature.zero_crossing_rate(audio)[0]
    
    # MFCC features
    mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=config['mfcc'])
    
    # Spectral entropy
    stft = np.abs(librosa.stft(audio))
    stft_norm = stft / (np.sum(stft, axis=0, keepdims=True) + 1e-8)
    spectral_entropy = -np.sum(stft_norm * np.log(stft_norm + 1e-8), axis=0)
    
    # High frequency energy ratio (>3kHz)
    freqs = librosa.fft_frequencies(sr=sr, n_fft=stft.shape[0]*2-1)
    hf_mask = freqs > 3000
    hf_energy = np.sum(stft[hf_mask[:stft.shape[0]], :], axis=0)
    total_energy = np.sum(stft, axis=0)
    hf_ratio = hf_energy / (total_energy + 1e-8)
    
    return {
        'spectral_centroid_mean': np.mean(spectral_centroids),
        'spectral_bandwidth_mean': np.mean(spectral_bandwidth),
        'spectral_rolloff_85': np.mean(spectral_rolloff),
        'spectral_flatness_mean': np.mean(spectral_flatness),
        'spectral_contrast_mean': np.mean(spectral_contrast),
        'zcr_mean': np.mean(zcr),
        'hf_energy_ratio': np.mean(hf_ratio),
        'spectral_entropy_mean': np.mean(spectral_entropy),
        'mfcc_summary': np.mean(mfccs, axis=1)  # Mean of each MFCC coefficient
    }

def extract_pitch_features(audio, sr):
    # Pitch tracking using librosa's yin
    f0 = librosa.yin(audio, fmin=50, fmax=500, sr=sr)
    
    # Voiced frames (non-zero f0)
    voiced_mask = f0 > 0
    pitch_presence_pct = np.sum(voiced_mask) / len(f0) * 100
    
    if np.any(voiced_mask):
        median_f0 = np.median(f0[voiced_mask])
        f0_var = np.var(f0[voiced_mask])
        
        # Simple jitter approximation (period-to-period variation)
        voiced_f0 = f0[voiced_mask]
        if len(voiced_f0) > 1:
            periods = 1.0 / (voiced_f0 + 1e-8)
            jitter = np.mean(np.abs(np.diff(periods))) / np.mean(periods) * 100
        else:
            jitter = 0
    else:
        median_f0 = 0
        f0_var = 0
        jitter = 0
    
    return {
        'pitch_presence_pct': pitch_presence_pct,
        'median_f0_hz': median_f0,
        'pitch_variation': f0_var,
        'jitter': jitter
    }

def extract_temporal_features(audio, sr):
    # RMS energy
    rms = librosa.feature.rms(y=audio)[0]
    rms_db = librosa.power_to_db(rms**2)
    
    # Modulation features (2-20 Hz modulation)
    if len(rms) > 20:  # Need minimum frames
        # Detrend RMS
        rms_detrended = signal.detrend(rms)
        # Simple modulation depth
        mod_depth = np.std(rms_detrended) / (np.mean(rms) + 1e-8)
    else:
        mod_depth = 0
    
    # Transient features
    energy = audio**2
    # Short-window energy for transient detection
    win_size = int(0.005 * sr)  # 5ms windows
    if len(energy) > win_size:
        windowed_energy = np.array([np.sum(energy[i:i+win_size]) 
                                  for i in range(0, len(energy)-win_size, win_size//2)])
        transient_kurtosis = stats.kurtosis(windowed_energy) if len(windowed_energy) > 3 else 0
    else:
        transient_kurtosis = 0
    
    return {
        'rms_db_mean': np.mean(rms_db),
        'rms_var': np.var(rms_db),
        'modulation_energy_low': mod_depth,
        'transient_kurtosis': transient_kurtosis
    }

def extract_advanced_features(audio, sr):
    """Extract advanced features."""
    # Harmonic-to-noise ratio approximation
    # Separate harmonic and percussive components
    harmonic, percussive = librosa.effects.hpss(audio)
    harmonic_energy = np.sum(harmonic**2)
    noise_energy = np.sum(percussive**2)
    hnr = 10 * np.log10((harmonic_energy + 1e-8) / (noise_energy + 1e-8))
    
    # Spectral features for texture
    stft = np.abs(librosa.stft(audio))
    
    # Simple roughness approximation (30-150 Hz modulation)
    if stft.shape[1] > 10:
        # Amplitude modulation in each frequency bin
        roughness_values = []
        for freq_bin in stft:
            if np.max(freq_bin) > 0:
                # Normalize
                norm_bin = freq_bin / np.max(freq_bin)
                # Simple modulation measure
                mod = np.std(norm_bin)
                roughness_values.append(mod)
        roughness = np.mean(roughness_values) if roughness_values else 0
    else:
        roughness = 0
    
    # Blood/crackling proxy (sudden broadband energy spikes)
    # Look for frames with sudden energy increase across many frequencies
    frame_energies = np.sum(stft, axis=0)
    if len(frame_energies) > 2:
        energy_diff = np.diff(frame_energies)
        sudden_spikes = np.sum(energy_diff > np.std(energy_diff) * 3)
        blood_proxy = sudden_spikes / len(energy_diff)
    else:
        blood_proxy = 0
    
    return {
        'harmonic_to_noise_ratio': hnr,
        'roughness': roughness,
        'blood_proxy_score': blood_proxy
    }

def extract_all_features(audio_path, config, augment=False):
    """Extract all 28 features from a single audio segment."""
    # Load audio
    audio, sr = librosa.load(audio_path, sr=config['sr'], mono=True)
    audio = torch.from_numpy(audio)#.unsqueeze(0)
    if random.uniform(0, 0.999) > 1 - 0.8 and augment:
        try:
            audio = data_augmentator(audio.unsqueeze(0), sr).squeeze(0)
        except:
            audio = audio

    #audio = (audio - wav_mean) / (wav_std + 1e-6)
    audio = 2 * (audio - audio.min()) / (audio.max() - audio.min() + 1e-6) - 1
    audio = audio.numpy()
    
    if len(audio) == 0:
        return None
    
    # Extract different feature groups
    envelope_features = extract_envelope_features(audio, sr)
    spectral_features = extract_spectral_features(audio, sr, config)
    pitch_features = extract_pitch_features(audio, sr)
    temporal_features = extract_temporal_features(audio, sr)
    advanced_features = extract_advanced_features(audio, sr)
    
    # Combine all features
    features = {
        'segment_path': audio_path,
        'segment_id': Path(audio_path).stem,
        **envelope_features,
        **spectral_features,
        **pitch_features,
        **temporal_features,
        **advanced_features
    }
    
    # Add individual MFCC features
    mfcc_summary = features.pop('mfcc_summary')
    for i, mfcc_val in enumerate(mfcc_summary):
        features[f'mfcc_{i+1}'] = mfcc_val
    
    return features

def compute_quantiles(df, feature_cols, percentiles=[20, 40, 60, 80]):
    """Compute quantile thresholds for each feature."""
    quantiles = {}
    
    for col in feature_cols:
        if col in df.columns:
            values = df[col].dropna()
            if len(values) > 0:
                # Compute percentiles
                thresholds = np.percentile(values, percentiles)
                quantiles[col] = thresholds.tolist()
            else:
                # Default thresholds if no valid data
                quantiles[col] = [0.0, 0.25, 0.5, 0.75]
        else:
            print(f"Warning: Feature '{col}' not found in data")
    
    return quantiles

def reduce_mfcc_features(df):
    """Reduce MFCC features using PCA to create composite features."""
    mfcc_cols = [f'mfcc_{i}' for i in range(1, 14)]
    mfcc_present = [col for col in mfcc_cols if col in df.columns]
    
    if len(mfcc_present) < 3:
        return df
    
    # Need sufficient samples for PCA
    if len(df) < 3:
        return df
    
    from sklearn.decomposition import PCA
    
    # Extract MFCC data
    mfcc_data = df[mfcc_present].fillna(0)
    
    # Apply PCA to reduce to min of (3, n_samples, n_features)
    n_components = min(3, len(df), len(mfcc_present))
    pca = PCA(n_components=n_components)
    mfcc_reduced = pca.fit_transform(mfcc_data)
    
    # Add reduced features to dataframe
    for i in range(mfcc_reduced.shape[1]):
        df[f'mfcc_pca_{i+1}'] = mfcc_reduced[:, i]
    
    print(f"MFCC PCA ({n_components} components) explained variance: {pca.explained_variance_ratio_}")
    
    return df

def create_derived_features(df):
    # Dryness (inverse of envelope modulation + low spectral variance)
    if 'envelope_mod_depth' in df.columns and 'spectral_flatness_mean' in df.columns:
        df['dryness'] = 1.0 / (df['envelope_mod_depth'] + df['spectral_flatness_mean'] + 1e-8)
    
    # Bubbliness (low frequency modulation + spectral irregularity)
    if 'modulation_energy_low' in df.columns and 'spectral_entropy_mean' in df.columns:
        df['bubbliness'] = df['modulation_energy_low'] * df['spectral_entropy_mean']
    
    # Wheeziness (high harmonic content in mid frequencies)
    if 'harmonic_to_noise_ratio' in df.columns and 'spectral_centroid_mean' in df.columns:
        # Wheeze typically in 400-1000 Hz range
        df['wheeziness'] = df['harmonic_to_noise_ratio'] * (1.0 / (np.abs(df['spectral_centroid_mean'] - 700) + 1))
    
    # Barkiness (sharp attack + mid-frequency emphasis)
    if 'attack_ms' in df.columns and 'spectral_centroid_mean' in df.columns:
        df['barkiness'] = (1.0 / (df['attack_ms'] + 1)) * (1.0 / (np.abs(df['spectral_centroid_mean'] - 650) + 1))
    
    # Hoarseness/Breathiness (high jitter + low harmonicity)
    if 'jitter' in df.columns and 'spectral_flatness_mean' in df.columns:
        df['hoarseness'] = df['jitter'] * df['spectral_flatness_mean']
        df['breathiness'] = df['spectral_flatness_mean'] * (1.0 / (df['harmonic_to_noise_ratio'] + 1e-8))
    
    # Crispness (low attack time + high transient content)
    if 'attack_ms' in df.columns and 'transient_kurtosis' in df.columns:
        df['crispness'] = df['transient_kurtosis'] / (df['attack_ms'] + 1)
    
    # Resonance (strong harmonic structure)
    if 'harmonic_to_noise_ratio' in df.columns and 'spectral_contrast_mean' in df.columns:
        df['resonance'] = df['harmonic_to_noise_ratio'] * df['spectral_contrast_mean']
    
    return df

def validate_quantiles(quantiles, df):
    """Validate that quantiles make sense."""
    issues = []
    
    for feature, thresholds in quantiles.items():
        if feature in df.columns:
            values = df[feature].dropna()
            if len(values) > 0:
                min_val, max_val = values.min(), values.max()
                
                # Check if thresholds are monotonic
                if not all(thresholds[i] <= thresholds[i+1] for i in range(len(thresholds)-1)):
                    issues.append(f"{feature}: Non-monotonic thresholds")
                
                # Check if thresholds span the data range reasonably
                if thresholds[0] > min_val or thresholds[-1] < max_val:
                    issues.append(f"{feature}: Thresholds don't span data range well")
    
    return issues

def load_quantiles(quantiles_path):
    """Load quantile thresholds."""
    with open(quantiles_path, 'r') as f:
        return json.load(f)

def numeric_to_categorical(value, thresholds):
    """Convert numeric value to categorical label using quantile thresholds."""
    if pd.isna(value):
        return 2  # Default to "Medium" for missing values
    
    for i, threshold in enumerate(thresholds):
        if value <= threshold:
            return i
    
    return len(thresholds)  # Highest category

def apply_quantiles_to_features(df, quantiles):
    """Apply quantile mapping to convert all numeric features to categorical."""
    categorical_df = df[['segment_path', 'segment_id']].copy()
    numeric_features = {}
    
    for feature, thresholds in quantiles.items():
        if feature in df.columns:
            # Store original numeric value
            numeric_features[feature] = df[feature]
            
            # Convert to categorical
            categorical_values = df[feature].apply(lambda x: numeric_to_categorical(x, thresholds))
            
            # Use human-readable name if available
            feature_name = CRITERIA_NAMES.get(feature, feature)
            categorical_df[f'{feature_name}_cat'] = categorical_values
    
    return categorical_df, numeric_features

def create_text_captions(categorical_df):
    """Create human-readable text captions for each segment."""
    captions = []
    
    for _, row in categorical_df.iterrows():
        segment_captions = {
            'segment_id': row['segment_id'],
            'segment_path': row['segment_path'],
            'captions': {}
        }
        
        # Convert categorical values to text descriptions
        for col in categorical_df.columns:
            if col.endswith('_cat') and col not in ['segment_id', 'segment_path']:
                criterion_name = col.replace('_cat', '')
                category_value = row[col]
                text_label = CATEGORY_LABELS.get(category_value, "Unknown")
                segment_captions['captions'][criterion_name] = text_label
        
        captions.append(segment_captions)
    
    return captions

def generate_summary_statistics(categorical_df, numeric_features):
    """Generate summary statistics for the captioning process."""
    stats = {
        'total_segments': len(categorical_df),
        'criteria_count': len([col for col in categorical_df.columns if col.endswith('_cat')]),
        'category_distributions': {},
        'numeric_feature_stats': {}
    }
    
    # Category distribution for each criterion
    for col in categorical_df.columns:
        if col.endswith('_cat'):
            criterion = col.replace('_cat', '')
            distribution = categorical_df[col].value_counts().to_dict()
            # Convert to percentages and readable labels
            total = sum(distribution.values())
            stats['category_distributions'][criterion] = {
                CATEGORY_LABELS.get(cat, f"Cat_{cat}"): count/total*100 
                for cat, count in distribution.items()
            }
    
    # Numeric feature statistics
    for feature, values in numeric_features.items():
        if len(values.dropna()) > 0:
            stats['numeric_feature_stats'][feature] = {
                'mean': float(values.mean()),
                'std': float(values.std()),
                'min': float(values.min()),
                'max': float(values.max()),
                'missing_pct': float(values.isna().mean() * 100)
            }
    
    return stats

def load_data(features_path):
    """Load features and labels data."""
    # Load features
    features_df = pd.read_parquet(features_path)
    return features_df

def prepare_features(features_df):
    """Prepare features for training."""
    # Extract categorical features (ending with '_cat')
    categorical_cols = [col for col in features_df.columns if col.endswith('_cat')]
    
    if not categorical_cols:
        print("Warning: No categorical features found, using all numeric features")
        feature_cols = [col for col in features_df.columns 
                       if col not in ['segment_path', 'segment_id'] 
                       and features_df[col].dtype in ['int64', 'float64', 'int32', 'float32']]
    else:
        feature_cols = categorical_cols
    
    # Extract feature matrix
    X = features_df[feature_cols].copy()
    
    # Handle missing values
    X = X.fillna(X.median())
    return X, feature_cols

def match_features_labels(features_df, labels_df):
    """Match features with labels based on file names."""
    # Extract filename from segment_path or segment_id
    if 'segment_path' in features_df.columns:
        features_df['filename'] = features_df['segment_path'].apply(
            lambda x: Path(x).name if isinstance(x, str) else None
        )
    elif 'segment_id' in features_df.columns:
        features_df['filename'] = features_df['segment_id'].apply(
            lambda x: f"{x}.wav" if isinstance(x, str) and not x.endswith('.wav') else x
        )
    else:
        raise ValueError("Neither segment_path nor segment_id found in features")

    # Match with labels - check for common column names
    filename_col = None
    for col in ['filename', 'file', 'audio_file', 'path_file']:
        if col in labels_df.columns:
            filename_col = col
            break

    if filename_col is None:
        print("Available columns in labels:", list(labels_df.columns))
        raise ValueError("Labels file must contain a filename column (filename, file, audio_file, or path_file)")

    # Extract just the filename from the labels path column
    if filename_col == 'path_file':
        labels_df['filename'] = labels_df[filename_col].apply(
            lambda x: Path(x).name if isinstance(x, str) else None
        )
    else:
        labels_df['filename'] = labels_df[filename_col]

    # Merge features with labels
    merged_df = features_df.merge(labels_df, on='filename', how='inner')
    print(f"Matched {len(merged_df)} samples with labels")

    # Verify participant column exists for group-based splitting
    if 'participant' not in merged_df.columns:
        print("Warning: 'participant' column not found in merged data")
        print(f"Available columns: {list(merged_df.columns)}")

    return merged_df

def train_model(X, y, model_type='xgboost', base_score=None, random_state=42):
    """Train the classification model."""
    
    if model_type == 'xgboost':
        print("Training XGBoost classifier...")
        model = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=random_state,
            eval_metric='logloss'
        )
        if base_score != None:
            model.set_params(base_score=0.32318032)
    else:
        print("Training Random Forest classifier...")
        model = RandomForestClassifier(
            n_estimators=100,
            max_depth=10,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=random_state
        )
    
    # Train model
    model.fit(X, y)
    
    return model

def evaluate_model(model, X, y, feature_names):
    """Evaluate model performance."""
    # Predictions
    y_pred = model.predict(X)
    y_pred_proba = model.predict_proba(X)
    
    # Handle single class case
    if y_pred_proba.shape[1] == 1:
        print("Warning: Model predicts only one class")
        y_pred_proba_pos = y_pred_proba[:, 0]  # Use the only available probability
    else:
        y_pred_proba_pos = y_pred_proba[:, 1]  # Positive class probability
    
    # Metrics
    metrics = {
        'accuracy': accuracy_score(y, y_pred),
        'precision': precision_score(y, y_pred, average='weighted', zero_division=0),
        'recall': recall_score(y, y_pred, average='weighted', zero_division=0),
        'f1': f1_score(y, y_pred, average='weighted', zero_division=0)
    }
    
    # Only compute AUC metrics if we have both classes
    unique_labels = np.unique(y)
    if len(unique_labels) > 1:
        try:
            # Ensure binary classification for ROC AUC
            if len(unique_labels) == 2:
                metrics['roc_auc'] = roc_auc_score(y, y_pred_proba_pos)
                precision_vals, recall_vals, _ = precision_recall_curve(y, y_pred_proba_pos)
                metrics['pr_auc'] = auc(recall_vals, precision_vals)
            else:
                print(f"Warning: Multi-class problem detected ({len(unique_labels)} classes), using weighted ROC AUC")
                metrics['roc_auc'] = roc_auc_score(y, y_pred_proba, multi_class='ovr', average='weighted')
                metrics['pr_auc'] = 0.0  # PR AUC not well-defined for multi-class
        except Exception as e:
            print(f"Warning: Error computing AUC metrics: {e}")
            metrics['roc_auc'] = 0.0
            metrics['pr_auc'] = 0.0
    else:
        print("Warning: Cannot compute AUC metrics with single class")
        metrics['roc_auc'] = 0.0
        metrics['pr_auc'] = 0.0
    
    # Feature importance
    if hasattr(model, 'feature_importances_'):
        feature_importance = pd.DataFrame({
            'feature': feature_names,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=False)
        
        print("\nTop 10 Most Important Features:")
        for _, row in feature_importance.head(10).iterrows():
            print(f"  {row['feature']}: {row['importance']:.4f}")
    
    return metrics

import numpy as np
import pandas as pd
import json

class QuantileBinner:
    """Quantile-based numeric-to-categorical transformer."""
    
    def __init__(self, percentiles=[20, 40, 60, 80], default_class=2):
        self.percentiles = percentiles
        self.default_class = default_class
        self.quantiles = {}

    def fit(self, df, feature_cols):
        """Compute quantile thresholds for each numeric feature."""
        for col in feature_cols:
            if col in df.columns:
                values = df[col].dropna()
                if len(values) > 0:
                    thresholds = np.percentile(values, self.percentiles)
                    self.quantiles[col] = thresholds.tolist()
                else:
                    self.quantiles[col] = [0.0, 0.25, 0.5, 0.75]
            else:
                print(f"Warning: Feature '{col}' not found in data")
        return self

    def _numeric_to_categorical(self, value, thresholds):
        """Internal converter for a single value."""
        if pd.isna(value):
            return self.default_class
        for i, threshold in enumerate(thresholds):
            if value <= threshold:
                return i
        return len(thresholds)

    def transform(self, df, criteria_names=None):
        """Apply fitted quantile bins to convert numeric features."""
        categorical_df = df[['segment_path', 'segment_id']].copy() if {'segment_path', 'segment_id'}.issubset(df.columns) else pd.DataFrame()
        
        for feature, thresholds in self.quantiles.items():
            if feature in df.columns:
                feature_name = criteria_names.get(feature, feature) if criteria_names else feature
                categorical_df[f"{feature_name}_cat"] = df[feature].apply(lambda x: self._numeric_to_categorical(x, thresholds))
        return categorical_df

    def save(self, path):
        with open(path, "w") as f:
            json.dump(self.quantiles, f, indent=2)

    def load(self, path):
        with open(path, "r") as f:
            self.quantiles = json.load(f)
        return self


In [ ]:
# Mapping from numeric features to the 28 human-interpretable criteria
FEATURE_MAPPING = {
    # Temporal features
    'pulse_count': 'Pulse_Count', 
    'inter_pulse_median': 'InterPulse_Median',
    'attack_ms': 'Attack_Time',
    'decay_ms': 'Decay_Time', 
    'attack_decay_ratio': 'Attack_Decay_Ratio',
    
    # Energy features
    'rms_db_mean': 'Loudness',
    'rms_var': 'RMS_Variation',
    'zcr_mean': 'Zero_Crossing_Rate',
    'envelope_mod_depth': 'Envelope_Modulation',
    
    # Spectral features
    'spectral_centroid_mean': 'Spectral_Brightness',
    'spectral_bandwidth_mean': 'Spectral_Bandwidth',
    'spectral_rolloff_85': 'Spectral_Rolloff',
    'spectral_flatness_mean': 'Harmonicity',  # Inverse relationship
    'harmonic_to_noise_ratio': 'Tonal_Clarity',
    'hf_energy_ratio': 'High_Freq_Content',
    'spectral_entropy_mean': 'Noise_Content',
    'spectral_contrast_mean': 'Spectral_Contrast',
    
    # Pitch features  
    'pitch_presence_pct': 'Pitch_Presence',
    'median_f0_hz': 'Fundamental_Freq',
    'pitch_variation': 'Pitch_Variation',
    'jitter': 'Pitch_Jitter',
    
    # Modulation features
    'modulation_energy_low': 'Low_Freq_Modulation',
    'transient_kurtosis': 'Transient_Sharpness',
    'roughness': 'Roughness',
    
    # Complex features
    'blood_proxy_score': 'Crackling_Proxy'
}

# Add MFCC features (will be combined using PCA)
for i in range(1, 14):  # MFCC 1-13
    FEATURE_MAPPING[f'mfcc_{i}'] = f'MFCC_{i}'

# Mapping of categorical labels
CATEGORY_LABELS = {
    0: "Very Low",
    1: "Low", 
    2: "Medium",
    3: "High",
    4: "Very High"
}

# Human-readable names for the 28 criteria
CRITERIA_NAMES = {
    'pulse_count': 'Pulse_Count',
    'inter_pulse_median': 'InterPulse_Interval', 
    'attack_ms': 'Attack_Sharpness',
    'decay_ms': 'Decay_Smoothness',
    'attack_decay_ratio': 'Attack_Decay_Balance',
    'rms_db_mean': 'Loudness',
    'rms_var': 'Loudness_Variation',
    'zcr_mean': 'Zero_Crossing_Rate',
    'envelope_mod_depth': 'Envelope_Modulation',
    'spectral_centroid_mean': 'Spectral_Brightness',
    'spectral_bandwidth_mean': 'Spectral_Bandwidth',
    'spectral_rolloff_85': 'Spectral_Rolloff',
    'spectral_flatness_mean': 'Noise_Content',
    'harmonic_to_noise_ratio': 'Harmonicity',
    'hf_energy_ratio': 'High_Frequency_Content',
    'spectral_entropy_mean': 'Spectral_Entropy',
    'spectral_contrast_mean': 'Spectral_Contrast',
    'pitch_presence_pct': 'Pitch_Presence',
    'median_f0_hz': 'Fundamental_Frequency',
    'pitch_variation': 'Pitch_Variation',
    'jitter': 'Pitch_Jitter',
    'modulation_energy_low': 'Low_Freq_Modulation',
    'transient_kurtosis': 'Crispness',
    'roughness': 'Roughness',
    'blood_proxy_score': 'Crackling_Content',
    
    # Derived features that map to clinical descriptions
    'dryness': 'Dryness',
    'bubbliness': 'Bubbliness', 
    'wheeziness': 'Wheeziness',
    'barkiness': 'Barkiness',
    'hoarseness': 'Hoarseness',
    'breathiness': 'Breathiness',
    'crispness': 'Perceived_Crispness',
    'resonance': 'Resonance',
    
    # PCA-reduced MFCC features
    'mfcc_pca_1': 'Spectral_Shape_1',
    'mfcc_pca_2': 'Spectral_Shape_2', 
    'mfcc_pca_3': 'Spectral_Shape_3'
}

In [18]:
df_train_ukcovid = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/ukcovid19/metadata.csv.train')
df_test_ukcovid = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/ukcovid19/metadata.csv.val')
df_train_ukcovid = df_train_ukcovid.rename(columns={"file_path": "path_file", 'covid_test_result': 'disease_status', 'gender': 'sex', 'participant_identifier': 'participant'})
df_test_ukcovid = df_test_ukcovid.rename(columns={"file_path": "path_file", 'covid_test_result': 'disease_status', 'gender': 'sex', 'participant_identifier': 'participant'})
df_2 = df_train_ukcovid[df_train_ukcovid['disease_status'] == 2]
df_4 = df_train_ukcovid[df_train_ukcovid['disease_status'] == 4]
df_2_down = resample(df_2, replace=False, n_samples=9138, random_state=42)
df_4_down = resample(df_4, replace=False, n_samples=6000, random_state=42)
df_train_ukcovid = pd.concat([df_2_down, df_4_down]).sample(frac=1, random_state=42).reset_index(drop=True)
df_2 = df_test_ukcovid[df_test_ukcovid['disease_status'] == 2]
df_4 = df_test_ukcovid[df_test_ukcovid['disease_status'] == 4]
df_2_down = resample(df_2, replace=False, n_samples=3682, random_state=42)
df_4_down = resample(df_4, replace=False, n_samples=1800, random_state=42)
df_test_ukcovid = pd.concat([df_2_down, df_4_down]).sample(frac=1, random_state=42).reset_index(drop=True)

df_longi = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/coda/longitudinal_original.csv')
df_longi = (
    df_longi.groupby('participant', group_keys=False)
    .apply(lambda x: x.sample(n=50, random_state=42) if len(x) > 50 else x)
)
df_longi = df_longi.rename(columns={"tb_status": "disease_status"})
df_0 = df_longi[df_longi['disease_status'] == 0]
df_1 = df_longi[df_longi['disease_status'] == 1]
df_0_down = resample(df_0, replace=False, n_samples=3000, random_state=42)
df_longi = pd.concat([df_0_down, df_1]).sample(frac=1, random_state=42).reset_index(drop=True)

df_solic = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/coda/solicited_original.csv')
df_solic = df_solic.rename(columns={"tb_status": "disease_status"})
df_0 = df_solic[df_solic['disease_status'] == 0]
df_1 = df_solic[df_solic['disease_status'] == 1]
df_0_down = resample(df_0, replace=False, n_samples=1800, random_state=42)
df_solic = pd.concat([df_0_down, df_1]).sample(frac=1, random_state=42).reset_index(drop=True)


df_tbscreen = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/TBscreen_Dataset/metadata_longitudinal.csv')
df_tbscreen = (
    df_tbscreen.groupby('participant', group_keys=False)
    .apply(lambda x: x.sample(n=50, random_state=42) if len(x) > 50 else x)
)

df_train_tbscreen = df_tbscreen[df_tbscreen['split'] == 'train'].reset_index(drop=True)
df_test_tbscreen   = df_tbscreen[df_tbscreen['split'] == 'validation'].reset_index(drop=True)

cols = ["path_file", "disease_status", "sex", "participant"]
df_longi = df_longi[cols]
df_solic = df_solic[cols]

df_train_tbscreen = df_train_tbscreen[cols]
df_test_tbscreen = df_test_tbscreen[cols]

df_train_ukcovid = df_train_ukcovid[cols]
df_test_ukcovid = df_test_ukcovid[cols]

df_train = pd.concat([df_longi, df_train_tbscreen, df_train_ukcovid], ignore_index=True)
df_test  = pd.concat([df_solic, df_test_tbscreen, df_test_ukcovid], ignore_index=True)

df_train['sex'] = df_train['sex'].replace({'Male': 0, 'Female': 1})
df_test['sex']  = df_test['sex'].replace({'Male': 0, 'Female': 1})

participant_mapping_longi = {participant: idx for idx, participant in enumerate(set(np.concatenate([df_train['participant'].unique(), df_test['participant'].unique()])))}
df_train['participant'] = df_train['participant'].map(participant_mapping_longi)
df_test['participant'] = df_test['participant'].map(participant_mapping_longi)

df_train['disease_status'] = df_train['disease_status'].replace(4, 0)
df_test['disease_status'] = df_test['disease_status'].replace(4, 0)

pickle_path = 'wav_stats.pickle'
if not os.path.exists(pickle_path):
    means, stds = [], []
    paths = df_train['path_file'].dropna().tolist()

    for path in tqdm(paths, desc="Processing WAV files", unit="file"):
        if not os.path.isfile(path):
            continue
        try:
            audio, _ = librosa.load(path, sr=None, mono=True)
            means.append(np.mean(audio))
            stds.append(np.std(audio))
        except Exception:
            continue

    stats_wavs = {
        "mean_db": float(np.mean(means)),
        "std_db": float(np.mean(stds))
    }

    with open(pickle_path, 'wb') as f:
        pickle.dump(stats_wavs, f)

with open(f"wav_stats.pickle", 'rb') as f:
    stats_wavs = pickle.load(f)
    wav_mean, wav_std = stats_wavs["mean_db"], stats_wavs["std_db"]

/tmp/ipykernel_867074/2922041318.py:19: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=50, random_state=42) if len(x) > 50 else x)
/tmp/ipykernel_867074/2922041318.py:38: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=50, random_state=42) if len(x) > 50 else x)
/tmp/ipykernel_867074/2922041318.py:57: FutureWarning: Downcasting behavior in `replace` is deprecated a

In [ ]:
from tqdm import tqdm

In [ ]:
config = load_config()
audio_files = df_train['path_file'].dropna().tolist()
all_features = []

for i, audio_file in enumerate(tqdm(audio_files, desc="Processing audio files")):
    features = extract_all_features(str(audio_file), config, True)
    if features:
        all_features.append(features)

features_df = pd.DataFrame(all_features)
features_df.to_parquet("/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/preprocessed_data/features_train.parquet", index=False)

audio_files = df_test['path_file'].dropna().tolist()
all_features = []

for i, audio_file in enumerate(tqdm(audio_files, desc="Processing audio files")):
    try:
        features = extract_all_features(str(audio_file), config, False)
        if features:
            all_features.append(features)
    except Exception as e:
        print(f"Error processing {audio_file}: {e}")

features_df = pd.DataFrame(all_features)
features_df.to_parquet("/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/preprocessed_data/features_test.parquet", index=False)

In [19]:
config = load_config()
percentiles = config['quantile_mapping']
df_features_train = pd.read_parquet("/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/preprocessed_data/features_train.parquet")
df_features_test = pd.read_parquet("/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/preprocessed_data/features_test.parquet")

df_features_train = create_derived_features(df_features_train)
df_features_test = create_derived_features(df_features_test)

df_features_train = reduce_mfcc_features(df_features_train)
df_features_test = reduce_mfcc_features(df_features_test)

numeric_features = [col for col in df_features_train.columns if col not in ['segment_path', 'segment_id'] and 
                    df_features_train[col].dtype in ['float64', 'int64', 'float32', 'int32']]

binner = QuantileBinner(percentiles=[20, 40, 60, 80])
binner.fit(df_features_train, numeric_features)
binner.save("/run/media/fourier/Data1/Pras/Thesis_Nexus/NXSCough/preprocessed_data/quantiles.json")

# categorical_df_train = binner.transform(df_features_train, CRITERIA_NAMES)
# categorical_df_test = binner.transform(df_features_test, CRITERIA_NAMES)

labels_df = df_train[["path_file", "disease_status", "participant"]].copy()
labels_df = labels_df.rename(columns={'path_file': 'segment_path'})

#df_train = categorical_df_train.merge(labels_df, on='segment_path', how='inner')
df_train = df_features_train.merge(labels_df, on='segment_path', how='inner')
df_train['disease_status'] = df_train['disease_status'].replace(2, 0)

feature_cols = [col for col in df_train.columns if col not in ['segment_id', 'segment_path', 'disease_status', 'participant']]
X = df_train[feature_cols].copy()
X = X.fillna(X.median())
y = df_train["disease_status"]

y = y.astype(int)
unique_vals = sorted(np.unique(y))
y = pd.Series(y.values) if hasattr(y, 'values') else pd.Series(y)

groups = df_train['participant'].values
n_participants = len(np.unique(groups))

if groups is not None:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))

    X_train = X.iloc[train_idx]
    X_dev = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_dev = y.iloc[test_idx]
    groups_train = groups[train_idx]
    groups_test = groups[test_idx]

    train_participants = set(groups_train)
    test_participants = set(groups_test)
    overlap = train_participants.intersection(test_participants)

model = train_model(X_train, y_train, 'xgboost')
model = train_model(X_train, y_train, 'xgboost', base_score=0.32318032)

test_metrics = evaluate_model(model, X_dev, y_dev, numeric_features)

print(f"\nTest Set Performance:")
for metric, score in test_metrics.items():
    print(f"  {metric}: {score:.4f}")

MFCC PCA (3 components) explained variance: [0.51710427 0.14439882 0.10611517]
MFCC PCA (3 components) explained variance: [0.60979986 0.10405711 0.0988679 ]
Training XGBoost classifier...
Training XGBoost classifier...

Top 10 Most Important Features:
  mfcc_1: 0.1061
  bubbliness: 0.0885
  mfcc_pca_3: 0.0769
  modulation_energy_low: 0.0599
  mfcc_pca_1: 0.0566
  resonance: 0.0405
  mfcc_pca_2: 0.0385
  mfcc_8: 0.0278
  spectral_contrast_mean: 0.0222
  attack_ms: 0.0215

Test Set Performance:
  accuracy: 0.7969
  precision: 0.7956
  recall: 0.7969
  f1: 0.7962
  roc_auc: 0.8855
  pr_auc: 0.7258


In [21]:
labels_df = df_test[["path_file", "disease_status", "participant"]].copy()
labels_df = labels_df.rename(columns={'path_file': 'segment_path'})

df_test = df_features_test.merge(labels_df, on='segment_path', how='inner')
df_test['disease_status'] = df_test['disease_status'].replace(2, 0)

X_test = df_test[feature_cols].copy()
X_test = X_test.fillna(X_test.median())
y_test = df_test["disease_status"]

y_test = y_test.astype(int)
y_test = pd.Series(y_test.values) if hasattr(y_test, 'values') else pd.Series(y_test)

test_metrics = evaluate_model(model, X_test, y_test, numeric_features)

print(f"\nTest Set Performance:")
for metric, score in test_metrics.items():
    print(f"  {metric}: {score:.4f}")


Top 10 Most Important Features:
  mfcc_1: 0.1061
  bubbliness: 0.0885
  mfcc_pca_3: 0.0769
  modulation_energy_low: 0.0599
  mfcc_pca_1: 0.0566
  resonance: 0.0405
  mfcc_pca_2: 0.0385
  mfcc_8: 0.0278
  spectral_contrast_mean: 0.0222
  attack_ms: 0.0215

Test Set Performance:
  accuracy: 0.7554
  precision: 0.7549
  recall: 0.7554
  f1: 0.7551
  roc_auc: 0.8524
  pr_auc: 0.6580


In [22]:
# Inference
config = load_config()

In [24]:
numeric_features

['pulse_count',
 'inter_pulse_median',
 'attack_ms',
 'decay_ms',
 'attack_decay_ratio',
 'envelope_mod_depth',
 'spectral_centroid_mean',
 'spectral_bandwidth_mean',
 'spectral_rolloff_85',
 'spectral_flatness_mean',
 'spectral_contrast_mean',
 'zcr_mean',
 'hf_energy_ratio',
 'spectral_entropy_mean',
 'pitch_presence_pct',
 'median_f0_hz',
 'pitch_variation',
 'jitter',
 'rms_db_mean',
 'rms_var',
 'modulation_energy_low',
 'transient_kurtosis',
 'harmonic_to_noise_ratio',
 'roughness',
 'blood_proxy_score',
 'mfcc_1',
 'mfcc_2',
 'mfcc_3',
 'mfcc_4',
 'mfcc_5',
 'mfcc_6',
 'mfcc_7',
 'mfcc_8',
 'mfcc_9',
 'mfcc_10',
 'mfcc_11',
 'mfcc_12',
 'mfcc_13',
 'dryness',
 'bubbliness',
 'wheeziness',
 'barkiness',
 'hoarseness',
 'breathiness',
 'crispness',
 'resonance',
 'mfcc_pca_1',
 'mfcc_pca_2',
 'mfcc_pca_3']

In [25]:
# df_infer = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/coda/solicited_original.csv')
# df_infer = df_infer.rename(columns={"tb_status": "disease_status"})
# df_infer = df_infer[["path_file", "disease_status", "sex", "participant"]]

# audio_files = df_infer['path_file'].dropna().tolist()
# all_features = []
# for i, audio_file in enumerate(tqdm(audio_files, desc="Processing audio files")):
#     features = extract_all_features(str(audio_file), config, True)
#     if features:
#         all_features.append(features)
# features_infer = pd.DataFrame(all_features)

# features_infer = create_derived_features(features_infer)
# features_infer = reduce_mfcc_features(features_infer)
# #categorical_features_infer = binner.transform(features_infer, CRITERIA_NAMES)

# labels_df = df_infer[["path_file", "disease_status", "participant"]].copy()
# labels_df = labels_df.rename(columns={'path_file': 'segment_path'})

# #df_infer = categorical_features_infer.merge(labels_df, on='segment_path', how='inner')
# df_infer = features_infer.merge(labels_df, on='segment_path', how='inner')
# df_infer['disease_status'] = df_infer['disease_status'].replace(2, 0)

feature_cols = [col for col in df_train.columns if col not in ['segment_id', 'segment_path', 'disease_status', 'participant']]#[col for col in df_infer.columns if col.endswith('_cat')]
X_infer = df_infer[feature_cols].copy()
X_infer = X_infer.fillna(X_infer.median())
y_infer = df_infer["disease_status"]
y_infer = pd.Series(y_infer.values) if hasattr(y_infer, 'values') else pd.Series(y_infer)

test_metrics = evaluate_model(model, X_infer, y_infer, numeric_features)
print(f"\nTest Set Performance:")
for metric, score in test_metrics.items():
    print(f"  {metric}: {score:.4f}")


Top 10 Most Important Features:
  mfcc_1: 0.1061
  bubbliness: 0.0885
  mfcc_pca_3: 0.0769
  modulation_energy_low: 0.0599
  mfcc_pca_1: 0.0566
  resonance: 0.0405
  mfcc_pca_2: 0.0385
  mfcc_8: 0.0278
  spectral_contrast_mean: 0.0222
  attack_ms: 0.0215

Test Set Performance:
  accuracy: 0.5016
  precision: 0.5988
  recall: 0.5016
  f1: 0.5215
  roc_auc: 0.5296
  pr_auc: 0.3255


In [ ]:
df_infer = pd.read_csv('/run/media/fourier/Data1/Pras/DatabaseLLM/cirdz/metadata_wavs.csv')
df_infer = df_infer.rename(columns={"tb_status": "disease_status"})
df_infer = df_infer[["path_file", "disease_status", "sex", "participant"]]

audio_files = df_infer['path_file'].dropna().tolist()
all_features = []
for i, audio_file in enumerate(tqdm(audio_files, desc="Processing audio files")):
    features = extract_all_features(str(audio_file), config, True)
    if features:
        all_features.append(features)
features_infer = pd.DataFrame(all_features)

features_infer = create_derived_features(features_infer)
features_infer = reduce_mfcc_features(features_infer)
categorical_features_infer = binner.transform(features_infer, CRITERIA_NAMES)

labels_df = df_infer[["path_file", "disease_status", "participant"]].copy()
labels_df = labels_df.rename(columns={'path_file': 'segment_path'})

df_infer = categorical_features_infer.merge(labels_df, on='segment_path', how='inner')
df_infer['disease_status'] = df_infer['disease_status'].replace(2, 0)

feature_cols = [col for col in df_infer.columns if col.endswith('_cat')]
X_infer = df_infer[feature_cols].copy()
X_infer = X_infer.fillna(X_infer.median())
y_infer = df_infer["disease_status"]
y_infer = pd.Series(y_infer.values) if hasattr(y_infer, 'values') else pd.Series(y_infer)

test_metrics = evaluate_model(model, X_infer, y_infer, numeric_features)
print(f"\nTest Set Performance:")
for metric, score in test_metrics.items():
    print(f"  {metric}: {score:.4f}")

In [ ]:
# =============================================================
# TBScreen
# =============================================================
hps.data.column_order = ["path_file", "disease_status", "sex", "participant"]
df = df_test_tbscreen
participant_mapping_longi = {participant: idx for idx, participant in enumerate(set(np.concatenate([df['participant'].unique()])))}
df['participant'] = df['participant'].map(participant_mapping_longi)
gender_mapping_longi = {gender: idx for idx, gender in enumerate(df['sex'].unique())}
df['sex'] = df['sex'].map(gender_mapping_longi)
df_test = df_test_tbscreen

collate_fn = CoughDatasetsCollate(hps.data.many_class)
val_dataset = CoughDatasets(df_test.values, hps.data, train=False)
val_loader = DataLoader(val_dataset, num_workers=28, shuffle=False, batch_size=hps.train.batch_size, pin_memory=True, drop_last=True, collate_fn=collate_fn)

cirdz_metrics = evaluate_model(val_loader, "tbscreen_solicited")

with open(f"{model_dir}/result_summary.txt", "a") as f:
    f.write(
        f"==================================== TBScreen Solicited =====================================\n"
    )
    f.write(
        f"Val - Acc {cirdz_metrics[0]:.2f} | BalAcc {cirdz_metrics[1]:.2f} | "
        f"Sens {cirdz_metrics[2]:.2f} | Spec {cirdz_metrics[3]:.2f}\n\n"
    )
